# NYUv2 Semantic Segmentation with RGB-D UNet

This project implements semantic segmentation on the NYUv2 dataset using a UNet architecture with 4-channel input (RGB + Depth).

## Key Design Choices
- Unified 256×256 resolution
- NEAREST interpolation for label resizing
- CrossEntropy + Dice loss combination
- Ignore index handling
- Mixed Precision (AMP)
- CosineAnnealing scheduler
- TTA (horizontal flip)

## Results
- Best validation mIoU: ~0.31
- Final TTA mIoU: 0.404

Depth information is concatenated as a 4th channel without handcrafted fusion, allowing the network to learn geometric representations directly.

#Configuration

In [ ]:
import os

image_dir = os.path.join(DATA_ROOT, "images")
mask_dir = os.path.join(DATA_ROOT, "masks")

['train', 'test']
['label', 'image', 'depth']
['000063.png', '000967.png', '000587.png', '000158.png', '000418.png']


# import library

In [ ]:
import os
from tqdm import tqdm
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch import optim
import torch.utils.data as data
from torch.utils.data import random_split, DataLoader
from torchvision.datasets import VisionDataset
from torchvision import transforms
from torchvision.transforms import (
    Compose,
    ColorJitter,
    Resize,
    ToTensor,
    Normalize,
    Lambda,
    InterpolationMode
)
from dataclasses import dataclass
import random

In [ ]:
# ===== Loss & Metrics =====
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0, ignore_index=255):
        super().__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        preds = torch.softmax(logits, dim=1)

        mask = targets != self.ignore_index
        targets = targets.clone()
        targets[~mask] = 0

        one_hot = torch.nn.functional.one_hot(
            targets, num_classes
        ).permute(0, 3, 1, 2)

        preds = preds * mask.unsqueeze(1)
        one_hot = one_hot * mask.unsqueeze(1)

        intersection = (preds * one_hot).sum(dim=(0, 2, 3))
        union = preds.sum(dim=(0, 2, 3)) + one_hot.sum(dim=(0, 2, 3))

        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


def mean_iou(logits, targets, num_classes, ignore_index=255):
    preds = torch.argmax(logits, dim=1)
    ious = []

    for cls in range(num_classes):
        if cls == ignore_index:
            continue
        pred_inds = preds == cls
        target_inds = targets == cls
        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()
        if union > 0:
            ious.append(intersection / union)

    if len(ious) == 0:
        return 0.0
    return sum(ious) / len(ious)


# DataLoader

In [ ]:
def colormap(N=256, normalized=False):
    def bitget(byteval, idx):
        return ((byteval & (1 << idx)) != 0)

    dtype = 'float32' if normalized else 'uint8'
    cmap = np.zeros((N, 3), dtype=dtype)
    for i in range(N):
        r = g = b = 0
        c = i
        for j in range(8):
            r = r | (bitget(c, 0) << 7-j)
            g = g | (bitget(c, 1) << 7-j)
            b = b | (bitget(c, 2) << 7-j)
            c = c >> 3
        cmap[i] = np.array([r, g, b])

    return cmap/255 if normalized else cmap


class NYUv2(VisionDataset):
    cmap = colormap()

    def __init__(
        self,
        root,
        split='train',
        include_depth=True,
        transform=None,
        target_transform=None
    ):
        super().__init__(root, transform=transform, target_transform=target_transform)

        assert split in ('train', 'test')
        self.root = root
        self.split = split
        self.include_depth = include_depth
        self.transform = transform
        self.target_transform = target_transform
            transforms.ToTensor(),
            transforms.Normalize(
               mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])


        self.depth_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])

        img_dir = os.path.join(self.root, self.split, 'image')
        img_names = sorted(os.listdir(img_dir))
        self.images = [os.path.join(img_dir, n) for n in img_names]

        if self.split == 'train':
            label_dir = os.path.join(self.root, self.split, 'label')
            self.targets = [os.path.join(label_dir, n) for n in img_names]

        depth_dir = os.path.join(self.root, self.split, 'depth')
        self.depths = [os.path.join(depth_dir, n) for n in img_names]

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        depth = Image.open(self.depths[idx])

        image = self.rgb_transform(image)
        depth = self.depth_transform(depth)

        if self.split == 'test':
            return image, depth

        target = Image.open(self.targets[idx])
        target = self.target_transform(target)

        return image, depth, target

    def __len__(self):
        return len(self.images)


# Model


In [ ]:
class NYUv2(VisionDataset):
    cmap = colormap()

    def __init__(self, root, split='train', include_depth=True):

        assert split in ('train', 'test')
        self.root = root
        self.split = split
        self.include_depth = include_depth
        self.target_transform = target_transform

        # RGB Transform
        self.rgb_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        # Depth Transform
        self.depth_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])

        img_dir = os.path.join(self.root, self.split, 'image')
        img_names = sorted(os.listdir(img_dir))
        self.images = [os.path.join(img_dir, n) for n in img_names]

        if self.split == 'train':
            label_dir = os.path.join(self.root, self.split, 'label')
            self.targets = [os.path.join(label_dir, n) for n in img_names]

        depth_dir = os.path.join(self.root, self.split, 'depth')
        self.depths = [os.path.join(depth_dir, n) for n in img_names]

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        depth = Image.open(self.depths[idx])

        image = self.rgb_transform(image)
        depth = self.depth_transform(depth)

        if self.split == 'test':
            return image, depth

        target = Image.open(self.targets[idx])
        target = target.resize((256, 256), Image.NEAREST)
        target = torch.from_numpy(np.array(target)).long()

        return image, depth, target

    def __len__(self):
        return len(self.images)

# Train and Valid

In [ ]:
# config
@dataclass
class TrainingConfig:
    dataset_root: str = "data"
    epochs: int = 40

    batch_size: int = 32
    num_workers: int = 4

    in_channels: int = 3
    num_classes: int = 13

    learning_rate: float = 0.001
    weight_decay: float = 1e-4

    train_val_split: float = 0.8

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    checkpoint_dir: str = "checkpoints"
    save_interval: int = 5

    image_size: tuple = (256, 256)
    normalize_mean: tuple = (0.485, 0.456, 0.406)
    normalize_std: tuple = (0.229, 0.224, 0.225)

    def __post_init__(self):
        import os
        os.makedirs(self.checkpoint_dir, exist_ok=True)

In [ ]:
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

#    Setup & Dataset

In [ ]:
set_seed(42)

config = TrainingConfig(
    dataset_root='./data',
    batch_size=16,
    num_workers=2,
    learning_rate=1e-4,
    epochs=100,
    image_size=(256, 256),
    in_channels=4
)

target_transform = Compose([
    Resize((256, 256), interpolation=InterpolationMode.NEAREST),
    Lambda(lambda lbl: torch.from_numpy(np.array(lbl)).long())
])

# ---- Dataset ----
full_dataset = NYUv2(root=config.dataset_root, split='train')

val_ratio = 0.1
n_total = len(full_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

train_dataset, val_dataset = random_split(
    full_dataset, [n_train, n_val]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers
)

# test dataset
test_dataset = NYUv2(
    root=config.dataset_root,
    split='test',
    include_depth=True,
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=config.num_workers
)

# shape sanity check
image, depth, label = next(iter(train_loader))
print(image.shape, depth.shape, label.shape)


torch.Size([16, 3, 256, 256]) torch.Size([16, 1, 256, 256]) torch.Size([16, 256, 256])


#    Model & Training

In [ ]:
device = config.device
print(f"Using device: {device}")

model = UNet(
    in_channels=config.in_channels,
    num_classes=config.num_classes
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.epochs
)

ce_loss = nn.CrossEntropyLoss(ignore_index=255)
dice_loss = DiceLoss(ignore_index=255)

from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

best_miou = 0.0
num_epochs = config.epochs

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    print(f"on epoch: {epoch+1}")

    with tqdm(train_loader) as pbar:
        for image, depth, label in pbar:
            image = image.to(device)
            depth = depth.to(device)
            label = label.to(device)

            optimizer.zero_grad()

            with autocast():
                x = torch.cat((image, depth), dim=1)
                pred = model(x)
                loss = ce_loss(pred, label) + dice_loss(pred, label)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

    scheduler.step()
    print(f"Epoch {epoch+1}, Train Loss: {total_loss / len(train_loader)}")

    # ---- Validation ----
    model.eval()
    val_miou = 0.0
    with torch.no_grad():
        for image, depth, label in val_loader:
            image = image.to(device)
            depth = depth.to(device)
            label = label.to(device)

            x = torch.cat((image, depth), dim=1)
            pred = model(x)
            val_miou += mean_iou(pred, label, config.num_classes)

    val_miou /= len(val_loader)
    print(f"Epoch {epoch+1}, val mIoU = {val_miou:.4f}")

    if val_miou > best_miou:
        best_miou = val_miou
        torch.save(model.state_dict(), "best_model.pth")


Using device: cuda
on epoch: 1


100%|██████████| 45/45 [00:32<00:00,  1.40it/s]

Epoch 1, Train Loss: 2.895898373921712


Epoch 1, val mIoU = 0.1128
on epoch: 2


100%|██████████| 45/45 [00:33<00:00,  1.33it/s]

...

Epoch 39, Train Loss: 0.9593625982602437


Epoch 39, val mIoU = 0.3108
on epoch: 40


100%|██████████| 45/45 [00:32<00:00,  1.40it/s]

Epoch 40, Train Loss: 0.9021076440811158


Epoch 40, val mIoU = 0.3141


#  Load best & TTA inference

In [ ]:
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

predictions = []
with torch.no_grad():
    print("Generating predictions with TTA...")
    for image, depth in tqdm(test_loader):
        image = image.to(device)
        depth = depth.to(device)

        # normal
        x = torch.cat((image, depth), dim=1)
        out1 = model(x)

        # flip TTA (horizontal)
        image_f = torch.flip(image, dims=[3])
        depth_f = torch.flip(depth, dims=[3])
        x_f = torch.cat((image_f, depth_f), dim=1)
        out2 = model(x_f)
        out2 = torch.flip(out2, dims=[3])  # flip back

        out = (out1 + out2) / 2.0
        pred = out.argmax(dim=1)  # [1, H, W]

        predictions.append(pred.cpu())

predictions = torch.cat(predictions, dim=0).numpy()
np.save("submission.npy", predictions)
print("Predictions saved to submission.npy")


Generating predictions with TTA...


100%|██████████| 654/654 [00:37<00:00, 17.23it/s]


Predictions saved to submission.npy
